# Workshop Quiz on Robotics Lesson 7 - "Robotic Perception and Control"

Modern robotics relies on precise **motion control** and **accurate environmental awareness**. **Control systems** and **state estimation** form the foundation of **intelligent autonomous behavior** by enabling robots to sense, interpret, and react to their surroundings effectively. In this module, we will explore three key concepts that make this possible:

- Feedback Control Systems – Ensuring robots maintain **desired motion** and **stability** by adjusting based on sensor data.
- State Estimation – Using mathematical models and sensor data to estimate a robot’s **true position**, **velocity**, and **orientation** despite measurement errors.
- Sensor Fusion – Combining multiple sensor inputs, such as IMUs and encoders, to improve **accuracy** and **robustness** in dynamic environments.

In [1]:
import module7
from module7 import functions_dict
def check_answer(question_id, charlie):
    if charlie.strip() == "":
        print("Please write your answer inside the quotations above.")
        return
    alpha = functions_dict.get(question_id, None)
    if alpha is None:
        print(f"Question ID '{question_id}' not found.")
    else:
        beta = alpha["answer"]
        if charlie.upper() == beta:
            print("Correct!")
        else:
            print("Incorrect. Hint:", alpha["hint"])

## **Feedback Control System**

Control systems are essential in robotics and engineering, allowing machines to **autonomously regulate** their behavior. Feedback control systems use sensor data to make real-time adjustments, ensuring **stable** and **accurate** performance in dynamic environments. 


### Why Are Control Systems Necessary?

Control systems are **fundamental** in robotics and engineering because they allow machines to function autonomously and reliably under various conditions. In many cases, systems are inherently **unstable** or **nonlinear**, meaning they do not behave predictably without constant adjustments. For example, a drone needs real-time control to maintain balance in mid-air, as any small disturbance (such as wind) can destabilize it. Similarly, an autonomous vehicle must track a path while accounting for road inclinations, friction, and sensor errors. 

Even for stable systems, control systems are used to enhance performance, making them more precise and efficient. Consider a conveyor belt that must maintain a steady speed despite variable loads. Without control, the system might slow down or speed up unpredictably. A well-designed control system ensures that external influences do not affect the desired operation of the system, improving reliability and efficiency.



### Open-Loop vs. Closed-Loop Control

Control systems can be categorized into **open-loop (feedforward)** and **closed-loop (feedback)** systems.

![open_closed_loops](OpenClosedLoops.png)

#### Open-Loop Control (Feedforward)

In an open-loop system, the **input** is sent directly to the system without measuring the **output**. The system assumes a fixed relationship between input and output. While simple and inexpensive, **open-loop control** is prone to **modeling errors** and **external disturbances**.

For example, if a motor is set to run at a certain speed using a predefined voltage, the system assumes that speed remains constant. However, if external forces (like friction or a change in load) affect the system, the motor speed may change without correction.


#### Closed-Loop Control (Feedback)

In a closed-loop system, **feedback** from sensors continuously **monitors** the system’s output and adjusts the input accordingly. A [Proportional integral derivative (PID) controller](https://en.wikipedia.org/wiki/Proportional%E2%80%93integral%E2%80%93derivative_controller) compares the **actual output** with the **desired setpoint** and **modifies** the input to reduce the error.

For example, in a self-balancing robot, gyroscopes measure tilt angles, and a controller adjusts motor speeds to maintain balance. Similarly, cruise control in a car adjusts throttle input based on real-time speed measurements to maintain a constant velocity.

#### Why Use Closed-Loop Control?

- Better stability – Ensures the system remains within a **safe range**.
- Disturbance rejection – **Compensates** for external factors like friction, wind, or terrain changes.
- Improved accuracy – Adjusts continuously to **minimize errors**.

For our rover project, we will be using closed-loop control with a rotary encoder to ensure the rover travels at a set speed, regardless of friction or terrain variations. By measuring wheel rotation, the controller can detect when the rover is moving too fast or too slow and adjust motor power accordingly.

### Question 1: Which of the following best describes the key difference between open-loop and closed-loop control systems?

**A.** Open-loop control relies on feedback, while closed-loop control does not. 

**B.** Closed-loop control continuously adjusts the system based on sensor feedback, while open-loop control does not use feedback and assumes a fixed relationship between input and output. 

**C.** Open-loop control is always more accurate than closed-loop control.

**D.** Closed-loop control does not require sensors to monitor system output. 

In [ ]:
answer_1 = ""

check_answer("Q1", answer_1)

### How to Design a Control System?

Designing a control system requires careful planning to ensure optimal performance. The key steps include:

1. Define Control Objectives – **Identify** the system’s **purpose** and required performance (e.g., maintain constant speed, balance, or precise positioning).
2. Select Sensors and Actuators – **Choose** appropriate **hardware** to measure and control the system’s behavior (e.g., rotary encoders for speed measurement, motors for motion).
3. Obtain a Mathematical Model – **Understand** how the system **responds** to inputs (e.g., how motor voltage affects wheel speed).
4. Design the Controller – **Implement** an algorithm that **adjusts** the system's behavior based on feedback (e.g., PID control).
5. Analyze and Optimize Performance – **Simulate** and test the system, making **adjustments** to improve stability and accuracy.

For our rover, we will:

- Use a rotary encoder to measure wheel speed.
- Implement a PID controller to adjust motor power dynamically.
- Ensure the rover maintains constant speed, regardless of external forces like incline or surface friction.

# **Feedback Control Systems Code**

### PID Control for Speed and Angle

**This section updates the speed and angle using Proportional-Integral-Derivative (PID) control.**
```
if (pidActive && (currentTime - lastPidUpdate >= pidInterval)) {
    float dt = (currentTime - lastPidUpdate) / 1000.0; // dt in seconds
    lastPidUpdate = currentTime;

    float currentOdom = get_odom();  // Get distance traveled
    currentSpeed = (currentOdom - lastodom) / dt; // v = s / t
    lastodom = currentOdom;

    currentAngle = get_angle_odom();  // Get orientation from odometry

// PID control for speed
    error_v = targetSpeed - currentSpeed;
    integral_v += error_v * dt;
    derivative_v = (error_v - lastError_v) / dt;
    output_v = (Kp_v * error_v + Ki_v * integral_v + Kd_v * derivative_v) * 0.25;
// 0.25 is a coefficient we added artificially to limit the size of the output. It is only valid for the robot we are currently using.
    lastError_v = error_v;

// PID control for angle
    error_a = targetAngle - currentAngle;
    integral_a += error_a * dt;
    derivative_a = (error_a - lastError_a) / dt;
    output_a = (Kp_a * error_a + Ki_a * integral_a + Kd_a * derivative_a) * 0.05;
// 0.05 is a coefficient we added artificially to limit the size of the output. It is only valid for the robot we are currently using.
    lastError_a = error_a;

// Generate motor control signals
    motor_controller(output_v + targetSpeed, output_a);
}
```



### Main PID Loop for Feedback Control

**This section updates speed and angle feedback, calculates PID errors, and adjusts motor control.**

```
if (pidActive && (currentTime - lastPidUpdate >= pidInterval)) {
    float dt = (currentTime - lastPidUpdate) / 1000.0; // dt in seconds
    lastPidUpdate = currentTime;

    float currentOdom = get_odom();  // Get distance traveled
    currentSpeed = (currentOdom - lastodom) / dt;
    lastodom = currentOdom;

    currentAngle = get_angle_odom();  // Get orientation from odometry

// PID control for speed
    error_v = targetSpeed - currentSpeed;
    integral_v += error_v * dt;
    derivative_v = (error_v - lastError_v) / dt;
    output_v = (Kp_v * error_v + Ki_v * integral_v + Kd_v * derivative_v) * 0.25;
    lastError_v = error_v;

// PID control for angle
    error_a = targetAngle - currentAngle;
    integral_a += error_a * dt;
    derivative_a = (error_a - lastError_a) / dt;
    output_a = (Kp_a * error_a + Ki_a * integral_a + Kd_a * derivative_a) / 20;
    lastError_a = error_a;

// Generate motor control signals
    motor_controller(output_v + targetSpeed, output_a);
}
```

### Motor Controller Function

**This function converts speed (v) and angular velocity (w) into left and right motor speeds.**

```
void motor_controller(float v, float w) {

// Reversed the velocity since forward is now backwards

    float dphi_L = -(v / r) + (L * w) / (2 * r);
    float dphi_R = -(v / r) - (L * w) / (2 * r);

    dphi_L = constrain(dphi_L, -11.52, 11.52);
    dphi_R = constrain(dphi_R, -11.52, 11.52);
  
    int duty_L = map(dphi_L, -11.52, 11.52, -255, 255);
    int duty_R = map(dphi_R, -11.52, 11.52, -255, 255);
  
    drive(duty_L, duty_R);
}
```

### Drive Function

**Controls the actual motor direction and PWM speed.**

```
void drive(int duty_L, int duty_R) {
    if (duty_L > 0) {
        digitalWrite(L1, HIGH);
        digitalWrite(L2, LOW);
    } else if (duty_L < 0) {
        digitalWrite(L1, LOW);
        digitalWrite(L2, HIGH);
    } else {
        digitalWrite(L1, LOW);
        digitalWrite(L2, LOW);
    }
  
    if (duty_R > 0) {
        digitalWrite(R1, HIGH);
        digitalWrite(R2, LOW);
    } else if (duty_R < 0) {
        digitalWrite(R1, LOW);
        digitalWrite(R2, HIGH);
    } else {
        digitalWrite(R1, LOW);
        digitalWrite(R2, LOW);
    }
  
    analogWrite(pwmL, abs(duty_L));
    analogWrite(pwmR, abs(duty_R));
}
```

### Odometry & Orientation Feedback

**These functions return distance traveled (get_odom()) and heading angle (get_angle_odom()).**

```
float get_odom() { 
// get encoder counts using getEncoderCount method from the Encoders class
    long left_encoder_count = left_encoder.getEncoderCount();
    long right_encoder_count = right_encoder.getEncoderCount();


// find the angular position of our wheels
    float left_wheel_pos = left_encoder_count * ((2 * 3.14) / 3575.04);
    float right_wheel_pos = right_encoder_count * ((2 * 3.14) / 3575.04);

// calculate the linear position of our robot from the angular position of the wheels
    float odom = (r / 2) * (left_wheel_pos + right_wheel_pos);
    return -odom; 
}

float get_angle_odom() {

// provided to make lesson 5 task more approachable  
// determines orientation of the robot using odometry and forward kinematics
// get encoder counts using getEncoderCount method from the Encoders class
   
    long left_encoder_count = left_encoder.getEncoderCount();
    long right_encoder_count = right_encoder.getEncoderCount();

    float left_wheel_pos = left_encoder_count * ((2 * 3.14) / 3575.04);
    float right_wheel_pos = right_encoder_count * ((2 * 3.14) / 3575.04);

    float heading = (r / L) * (right_wheel_pos - left_wheel_pos) * RAD_TO_DEG;
    return heading;
}
```

### Summary

This feedback control system:

- Reads sensor data from encoders (get_odom() and get_angle_odom()).
- Calculates errors between the target and current speed/angle.
- Uses PID controllers to generate speed (output_v) and angle (output_a) corrections.
- Converts control outputs into motor speed commands via motor_controller().
- Sends signals to motors using drive(), adjusting direction and speed via PWM.

This ensures that the rover maintains a desired speed and heading using real-time sensor feedback. With these aspects you have the necessary code to expand upon your current rover controller. Let's take your new controller and test this basic Feedback Control System.

# **Testing Feedback Control System**


#### Objective
These tests evaluate the rover's ability to regulate its speed and adjust movement dynamically using feedback control systems. Specifically, we will test **Turn Control** to ensuring the rover reduces speed while turning to prevent overshooting.



### Controlled 90-Degree Turn with Speed Regulation

#### Purpose:

To assess how well the rover adjusts its speed while turning to avoid overshooting beyond the 90-degree mark.

#### Setup:

- A 1m x 1m test area marked with a starting line and a 90-degree target line.
- The wheel encoder tracks wheel rotations, allowing the rover to compute how much it has turned.

#### Procedure:
1. Straight Movement Baseline:
    - The rover moves forward at a set speed.
2. Wheel Encoder Turn Calculation:
    - We estimate the turn using differential wheel movement.
        - Example: If we know a full wheel rotation moves the rover X cm, we calculate how much wheel movement corresponds to a 90-degree pivot.
3. Speed Reduction Before Completion:
    - As the rover nears the computed turn threshold, it reduces speed to prevent overshooting.
4. Final Verification:
    - The rover should stop turning exactly at 90 degrees, using the encoder readings to validate the stopping point.
    - If it overshoots or stops short, we refine the speed reduction algorithm.
    
#### Expected Results:
- The rover should stop at 90° ±2°.
- Encoder readings should accurately guide turning without requiring an IMU.
- Adjustments can be made by tuning the speed reduction timing based on real-world results.

## **Sensor Fusion**

Robots rely on sensors to perceive and understand their environment, but every sensor has **limitations**. Some sensors are **noisy**, some **drift** over time, and others have a **limited range**. Sensor fusion is the process of **combining data** from multiple sensors to improve accuracy and reliability. By fusing different sensor data, we can **compensate** for **individual weaknesses** and create a more accurate and stable representation of the robot’s state.

![SensorFusion](SensorFusion.png)

### Why Do We Need Sensor Fusion?

Each sensor we use has strengths and weaknesses:

1. IMU (Inertial Measurement Unit)
- ✅ Measures angular velocity and acceleration.
- ✅ Works in all lighting and environmental conditions.
- ❌ Drifts over time due to accumulated errors in integration.
- ❌ Cannot measure absolute position.
2. ToF (Time-of-Flight) Sensor
- ✅ Provides accurate absolute distance to objects.
- ✅ Works well for obstacle detection and range measurements.
- ❌ Cannot track lateral motion or orientation changes.
- ❌ Susceptible to errors in bright lighting or reflective surfaces.

By combining these two sensors, we can:

- Use IMU data for high-frequency motion tracking.
- Use ToF sensor data to periodically correct position estimates.
- Reduce drift and sensor noise using data filtering techniques (e.g., Kalman filters).


### Question 2: Why is sensor fusion important for improving a robot’s state estimation?

**A.** It helps compensate for individual sensor weaknesses by combining their strengths.

**B.** It eliminates the need for filtering or error correction. 

**C.** It allows the robot to function without any sensors.

**D.** It makes sensors work faster but less accurately.

In [ ]:
answer_2 = ""

check_answer("Q2", answer_2)

### Kalman Filtering

Kalman Filter (KF) is a powerful mathematical tool that allows us to **optimally combine** these sensor readings from above to improve overall accuracy.

A Kalman Filter is an algorithm that estimates the state of a dynamic system by:
- Predicting the next state using a motion model.
- Correcting the prediction using sensor measurements.
- Weighting each estimate based on uncertainty (trusting more reliable sources).

This process repeats continuously, refining the state estimate over time.



#### How the Kalman Filter Works

The Kalman Filter operates in two main steps: Prediction and Update (Correction).

1. **Prediction Step**
- Uses the previous state and a motion model to estimate the rover’s next position.
- Example: The IMU detects acceleration and rotation, predicting movement.

$$\hat{x}_{k|k-1} = A x_{k-1} + B u_k$$

Where:

- $\hat{x}_{k|k-1}$ = Predicted state at step $k$
- $A$ = State transition matrix (how the system evolves over time)
- $x_{k-1}$ = Previous state estimate
- $B u_k$ = Control input (e.g., wheel speed, acceleration)

2. **Update Step**

- Compares predicted state with sensor measurements (ToF sensor for position).
- Computes a correction factor (Kalman Gain, K) that balances trust between IMU and ToF data.
$$K = P H^T (H P H^T + R)^{-1}$$

$K$ (**Kalman Gain**): How much trust to place in the sensor vs. prediction.

$P$ (**State Uncertainty**): Uncertainty in predicted position.

$H$ (**Observation Model**): Relates sensor measurements to the state.

$R$ (**Sensor Noise**): Variability in sensor readings.


Finally, the new state estimate is computed:

$$x_k = \hat{x}_{k|k-1} + K (z_k - H \hat{x}_{k|k-1})$$

Where:

- $z_k$  is the sensor measurement (ToF reading)
- $x_k$ is the updated state estimate.

This ensures smooth, accurate position tracking despite sensor noise and IMU drift.

### Question 3: What is the primary advantage of using a Kalman Filter in sensor fusion?

**A.** It completely eliminates all sensor errors.

**B.** It predicts the system's state and corrects it using sensor data.

**C.** It only relies on one sensor at a time for estimation.

**D.** It predicts the system's state and corrects it using sensor data.

In [2]:
answer_3 = ""

check_answer("Q3", answer_3)

Please write your answer inside the quotations above.


### Kalman Filter in Our Rover

In the next step, State Estimation we will use a Kalman Filter in our system since the IMU provides **high-frequency motion updates** and the ToF sensor corrects **position errors** periodically. The Kalman Filter blends these sources to give the rover an accurate estimate of its position, even if sensors individually have flaws.

## **State Estimation**

In robotics, state estimation is crucial for understanding the robot's position, velocity, and orientation based on sensor data. Robots often rely on sensors like gyroscopes, accelerometers, encoders, and cameras to interpret their surroundings, but no sensor is perfect. Each sensor has strengths and weaknesses, and measurement errors can accumulate over time, leading to inaccurate positioning.

For example, a rover navigating a rough terrain might use wheel encoders to track its movement. However, if the wheels slip on loose dirt or ice, the encoder readings will not accurately reflect the actual distance traveled.

**State estimation techniques allow robots to combine multiple sensor readings to produce a more accurate estimate of their true position and movement**. By fusing sensor data, the system corrects for individual sensor inaccuracies and provides a more reliable representation of the robot’s state.

### Dead Reckoning vs. Absolute Positioning

There are two main approaches to estimating a robot’s state:

#### Dead Reckoning (Relative Estimation)

[Dead reckoning](https://aleksandarhaber.com/what-is-dead-reckoning-clear-explanation-with-python-simulation-part-i/) involves estimating the robot's position based on previous states. It integrates velocity and acceleration measurements over time to determine how far the robot has moved.

- Advantages : Works in environments where GPS or external references are unavailable (e.g., underground, inside buildings).
- Disadvantages : Errors accumulate over time due to sensor noise and drift, requiring corrections from external references.

#### Absolute Positioning

[Absolute positioning](https://www.robotsforroboticists.com/gps/) relies on external references, such as GPS, beacons, or pre-mapped landmarks, to directly measure the robot’s position.

- Advantages : Provides accurate long-term positioning without drift.
- Disadvantages : Requires external infrastructure (e.g., GPS satellites, vision-based markers).

### Question 4: A rover navigating indoors relies on dead reckoning using an IMU and a Time-of-Flight (ToF) sensor. What problem might occur if it only uses the IMU without any external corrections?

**A.** Small sensor errors will accumulate, causing the estimated position to drift over time.

**B.** The rover will perfectly track its position without errors.

**C.** The ToF sensor will become unnecessary.

**D.** The IMU will automatically correct its own errors.

In [ ]:
answer_4 = ""

check_answer("Q4", answer_4)

## **State Estimation Code**

### We have provided the code in accordance with this rover, but have left some [YOUR CODE HERE] in def ekf_callback()
```
from ekf import ExtendedKalmanFilter
import numpy as np
import time
import math

ekf = ExtendedKalmanFilter(state_dim=3, obs_dim=2)  # Changed to 2D measurement
ekf.Q = np.diag([0.05, 0.05, 0.01])  # Process noise: [x, y, theta]
ekf.R = np.diag([0.1, 0.1])          # Measurement noise for [x, y]

# Initialize variables
last_time = time.time()
initial_x = 0
initial_y = 0

def ekf_callback(ax, ay, wz, tof):
    global last_time
    current_time = time.time()
    dt = current_time - last_time
    last_time = current_time
    
    # Predict step using IMU data
    ekf.predict(acceleration=(ax, ay), 
                angular_vel=wz,
                dt=dt)
    
    # Update state if TOF measurement is valid
    if tof < 2.0:
        # Current robot state
        current_state = ekf.get_state()
        theta = current_state[2]  # Current heading
        
        # Convert TOF measurement to global coordinates
        # TOF measures distance in front of robot, so we need to project it
        # based on robot's current orientation
        # measured_x = [YOUR CODE HERE]
        # measured_y = [YOUR CODE HERE]
        
        # Observation matrix for both x and y
        H = np.array([
            [1, 0, 0],  # x measurement
            [0, 1, 0]   # y measurement
        ])
        
        # Update with converted measurements
        ekf.update(z=np.array([measured_x, measured_y]),
                  H=H,
                  R=np.diag([0.1, 0.1]))
    
    state = ekf.get_state()
    print(f"Estimated State: x={state[0]:.2f}m, y={state[1]:.2f}m, θ={np.degrees(state[2]):.1f}°")
    print(f"Raw TOF reading: {tof:.2f}m")  # Added for debugging

from send import process_serial_data
process_serial_data(callback=ekf_callback)
```

## Testing State Estimation: Obstacle Avoidance with IMU & ToF Fusion

For our rover, we will use an **Inertial Measurement Unit (IMU)** and a **Time-of-Flight (ToF) sensor** for state estimation. These sensors provide motion tracking and obstacle distance measurements, allowing the rover to estimate its position without depending on wheel odometry.

- **IMU** : Measures changes in orientation (yaw, pitch, and roll) and acceleration. This allows us to estimate rotational motion and overall movement direction.
- **ToF Sensor** : Measures the distance to nearby obstacles, helping correct errors and refine position estimates.

#### Objective

Evaluate the accuracy of state estimation when the rover encounters and avoids an obstacle, ensuring that the IMU (Inertial Measurement Unit) and ToF (Time of Flight) sensor work together to track movement.

#### Setup

1. Test Area:

- Mark a 1m x 1m test area with a clear start line and an endpoint.
- The terrain should be flat and free of unnecessary obstacles.

2. Obstacle Placement:
- Position a physical obstacle exactly 0.5 meter from the starting line, directly in the rover’s path.
- The obstacle should be tall enough to be detected by the ToF sensor but not block lateral movement.

#### Procedure

1. Initial Movement:

- Command the rover to move straight forward at a set speed.
- Log data from both the IMU (for motion tracking) and ToF (for obstacle detection).

2. Obstacle Detection & Avoidance:

- When the ToF sensor detects the obstacle at (X) meter distance, the rover should:
    - Turn 90° right (perpendicular to the original path).
    - Move (Y) meters sideways to bypass the obstacle.
    - Turn 90° left to return to the original direction.
- The rover should then continue forward until reaching the endpoint.

3. Final Position Analysis:

- Once the rover reaches the endpoint, compare its estimated final position (from sensor data) with the expected final position (2m forward, 0.5m lateral shift).
- Log the difference between estimated vs. actual position to assess drift and accuracy.

### Kalman Filter

1. Prediction Phase (Before Obstacle Detection)
- The rover starts moving forward using IMU-based motion tracking.
- Instead of relying solely on raw IMU readings, Kalman filtering predicts the rover’s future position.
- Expected Benefit: Reduces error buildup from IMU drift by combining acceleration and orientation data.
2. Correction Phase (Obstacle Detection)
- When the ToF sensor detects the obstacle, the Kalman filter corrects the position estimate based on ToF distance data.
- The filter updates the rover’s belief about its location before executing avoidance maneuvers.
- Expected Benefit: ToF data helps counteract position drift by periodically resetting the estimated location.

### Understand due to the limitations of the IMU there is most likely going to be a 5 degree error in this demo.

# Conclusion

In this module, we explored how feedback control, state estimation, and sensor fusion form the backbone of autonomous robotics. You have learned how to:

- Regulate speed and heading using PID control. 
- Blend sensor data through Kalman filtering.
- Estimate the robot’s position in real time.

You now have the essential tools to build reliable and intelligent motion. These techniques aren't just theoretical — they are used in drones, self-driving cars, and rovers just like yours. Mastering them puts you one step closer to full autonomy. Up next: applying this knowledge in SLAM to build maps and plan paths intelligently.

